# bdap_lea — notebook v0

Notebook v0 di validazione del mart in `dataset-incubator`.

- scopo: sanity check e lettura base del mart
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit


In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SLUG = "bdap_lea"
MART_TABLE = "mart_spesa_enti_2024"
METRICA = "importo_totale"

candidate_dir = Path.cwd()
if not (candidate_dir / "dataset.yml").exists():
    if (candidate_dir.parent / "dataset.yml").exists():
        candidate_dir = candidate_dir.parent
    else:
        raise FileNotFoundError(
            f"dataset.yml non trovato in {Path.cwd()} o nella cartella parent."
        )

mart_root = candidate_dir.parents[1] / "out" / "data" / "mart" / SLUG
mart_candidates = sorted(mart_root.glob(f"*/{MART_TABLE}.parquet"))
if not mart_candidates:
    raise FileNotFoundError(
        f"Mart non trovato per {SLUG}/{MART_TABLE} sotto {mart_root}. Eseguire prima toolkit run all."
    )

PARQUET_PATH = mart_candidates[-1]
print(f"Candidate dir: {candidate_dir.name}")
print(f"Mart table: {PARQUET_PATH.name}")


Candidate dir: bdap-lea
Mart table: mart_spesa_enti_2024.parquet


In [2]:
con = duckdb.connect()
df = con.execute("SELECT * FROM read_parquet(?)", [str(PARQUET_PATH)]).df()
print(f"Shape: {df.shape}")
display(df.dtypes)


Shape: (23321, 21)


anno_riferimento                    int32
codice_regione                        str
descrizione_regione                   str
codice_ente_ssn                       str
codice_ente_bdap                    int64
descrizione_ente                      str
codice_voce_contabile                 str
descrizione_voce_contabile            str
consumi_sanitari                  float64
consumi_non_sanitari              float64
prestazioni_sanitarie             float64
servizi_sanitari                  float64
servizi_non_sanitari              float64
personale_sanitario               float64
personale_professionale           float64
personale_tecnico                 float64
personale_amministrativo          float64
ammortamenti                      float64
sopravvenienze_e_insussistenze    float64
altri_costi                       float64
importo_totale                    float64
dtype: object

In [3]:
display(df.head(10))


,anno_riferimento,codice_regione,descrizione_regione,codice_ente_ssn,codice_ente_bdap,descrizione_ente,codice_voce_contabile,descrizione_voce_contabile,consumi_sanitari,consumi_non_sanitari,...,servizi_sanitari,servizi_non_sanitari,personale_sanitario,personale_professionale,personale_tecnico,personale_amministrativo,ammortamenti,sopravvenienze_e_insussistenze,altri_costi,importo_totale
0,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,19999,TOTALE PREVENZIONE COLLETTIVA E SANITA' PUBBLICA,4678820.17,309896.94,...,3198606.04,5123804.57,15482447.77,64292.70,2662804.14,4151991.15,590962.81,547075.00,1211670.32,40157781.45
1,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1A100,"Sorveglianza, prevenzione e controllo delle ma...",3918367.73,21786.88,...,375218.60,921903.17,1251507.41,5420.70,149029.05,458199.23,105028.22,8380.19,61542.93,7328482.08
2,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1A110,Vaccinazioni,3907184.75,20346.92,...,206396.71,615037.20,1235520.50,4598.54,119738.78,235580.51,53077.10,7172.19,52739.23,6509434.85
3,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1A120,"Altri interventi per la sorveglianza, prevenzi...",11182.98,1439.96,...,168821.89,306865.97,15986.91,822.16,29290.27,222618.72,51951.12,1208.00,8803.70,819047.23
4,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1B100,Tutela della salute e della sicurezza degli am...,38927.29,25730.79,...,120595.49,265181.43,1077782.67,3778.76,82967.92,144883.77,22465.40,5886.38,51022.43,1840106.55
5,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1C100,"Sorveglianza, prevenzione e tutela della salut...",41770.82,31189.83,...,150713.82,502578.17,1588747.37,6164.13,145367.09,388437.17,21829.85,9569.50,53573.02,2941400.51
6,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1D100,Salute animale e igiene urbana veterinaria,211525.86,75762.95,...,146515.16,876681.28,3901918.11,14768.74,529731.69,706854.91,39080.87,23166.20,138913.02,7131883.31
7,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1E100,Sicurezza alimentare - Tutela della salute dei...,121833.91,76979.44,...,100814.22,805824.16,3863151.47,13908.54,361244.78,597455.36,36784.25,21738.84,172337.88,6477254.33
8,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1F100,Sorveglianza e prevenzione delle malattie cron...,298422.04,49960.44,...,500754.18,1264495.70,3799340.74,20251.83,1394463.61,1856160.71,345879.13,31783.70,454243.77,10793786.63
9,2024,010,Piemonte,203,175142930473519802,AZIENDA SANITARIA LOCALE TO3,1F110,Screening oncologici,184348.07,15216.89,...,259273.38,229709.38,1135567.01,4421.99,172006.38,223477.65,50777.55,6902.62,240598.41,3039339.61


In [4]:
print("Null per colonna:")
display(df.isnull().sum())

if METRICA in df.columns:
    negativi = int((df[METRICA] < 0).sum()) if pd.api.types.is_numeric_dtype(df[METRICA]) else "n/a"
    print(
        f"\nRange {METRICA}: min={df[METRICA].min()}, max={df[METRICA].max()}, negativi={negativi}"
    )
else:
    print(f"Metrica non trovata nel mart: {METRICA}")


Null per colonna:


anno_riferimento                  0
codice_regione                    0
descrizione_regione               0
codice_ente_ssn                   0
codice_ente_bdap                  0
descrizione_ente                  0
codice_voce_contabile             0
descrizione_voce_contabile        0
consumi_sanitari                  0
consumi_non_sanitari              0
prestazioni_sanitarie             0
servizi_sanitari                  0
servizi_non_sanitari              0
personale_sanitario               0
personale_professionale           0
personale_tecnico                 0
personale_amministrativo          0
ammortamenti                      0
sopravvenienze_e_insussistenze    0
altri_costi                       0
importo_totale                    0
dtype: int64


Range importo_totale: min=0.0, max=24693630006.0, negativi=0


In [5]:
# DIM = "descrizione_regione"
# (
#     df.groupby(DIM)[METRICA]
#     .sum()
#     .sort_values(ascending=False)
#     .plot(kind="bar", figsize=(12, 5), title=f"{METRICA} per {DIM}")
# )
# plt.tight_layout()
# plt.show()
print("Cella visualizzazione opzionale: decommentare e adattare a descrizione_regione")


Cella visualizzazione opzionale: decommentare e adattare a descrizione_regione


## Note v0

- Slug: `bdap_lea`
- Tabella mart: `mart_spesa_enti_2024`
- Metrica guida: `importo_totale`
- Perimetro: solo 2024, enti operativi (`Codice Ente SSN != '000'`), lettura prudente della prevenzione collettiva
- Questo notebook resta esplorativo e validativo in DI.
